# GRID Gemma 4 Fine-Tuning with Unsloth

Fine-tune Gemma 4 (E2B / E4B / 26B-A4B / 31B) for GRID-specific tasks:
- **Signal Classifier**: Domain + urgency classification for incoming market signals
- **Anomaly Narrator**: One-line summaries of statistical anomalies
- **EDGAR Extractor**: Structured data extraction from SEC filings
- **Knowledge Mapper**: Wiki-style [[backlinks]] and hidden connection discovery

**Requirements**: GPU runtime (T4 for E2B, L4/A100 for E4B+)

| Model | VRAM (4-bit) | Colab Tier |
|-------|-------------|------------|
| Gemma 4 E2B | ~4 GB | Free T4 |
| Gemma 4 E4B | ~6 GB | Free T4 |
| Gemma 4 26B-A4B | ~18 GB | Pro A100 |
| Gemma 4 31B | ~20 GB | Pro A100 |

## 1. Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl==0.22.2

## 2. Clone GRID Repo (for dataset generators)

In [ ]:
import os

# Use HF token for gated model access
HF_TOKEN = os.environ.get("HF_TOKEN", "")
if not HF_TOKEN:
    # Try loading from .env if running locally
    try:
        with open("/content/grid/.env") as f:
            for line in f:
                if line.startswith("HF_API_KEY="):
                    HF_TOKEN = line.strip().split("=", 1)[1]
                    os.environ["HF_TOKEN"] = HF_TOKEN
                    break
    except FileNotFoundError:
        pass

if not os.path.exists('/content/grid'):
    !git clone https://github.com/3pacs/GRID.git /content/grid
    %cd /content/grid
else:
    %cd /content/grid
    !git pull

import sys
sys.path.insert(0, '/content/grid')

print(f"HF Token: {'configured' if HF_TOKEN else 'NOT SET — may fail on gated models'}")

## 3. Configuration

Choose your task and model size. Gemma 4 E4B with QLoRA is the sweet spot — better than E2B at full precision.

In [ ]:
# === CONFIGURE HERE ===
TASK = "knowledge_mapper"  # signal_classifier | anomaly_narrator | edgar_extractor | knowledge_mapper
BASE_MODEL = "unsloth/gemma-4-E4B-it"  # or E2B for free Colab
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = True

# LoRA config — r=8 for small datasets (<1000), r=16 for larger
LORA_R = 8
LORA_ALPHA = 8  # Keep 1:1 ratio with r

# Training hyperparameters
EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4  # Effective batch = 2 * 4 = 8
LEARNING_RATE = 2e-4  # Reduce to 2e-5 for longer runs / larger datasets
WARMUP_STEPS = 10

# Export
EXPORT_GGUF = True
GGUF_QUANT = "q8_0"  # q8_0, q4_k_m, q5_k_m, f16, bf16

## 4. Load Model + Attach LoRA

In [ ]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_8bit=False,
    full_finetuning=False,
    dtype=None,  # Auto-detect
)

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=0,
    bias="none",
    random_state=3407,
)

# Print trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## 5. Prepare Dataset

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats
from gemma.training.config import TaskType, TASK_SYSTEM_PROMPTS
from gemma.training.datasets import load_dataset_for_training

# Apply Gemma chat template (gemma-3 template works for Gemma 4 — same structure)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-3")

# Load GRID task-specific dataset
task = TaskType(TASK)
dataset = load_dataset_for_training(task)
dataset = standardize_data_formats(dataset)

def format_conversations(examples):
    convos = examples["conversations"]
    texts = [
        tokenizer.apply_chat_template(
            convo, tokenize=False, add_generation_prompt=False
        ).removeprefix("<bos>")
        for convo in convos
    ]
    return {"text": texts}

dataset = dataset.map(format_conversations, batched=True)
print(f"Dataset: {len(dataset)} examples for task '{TASK}'")
print(f"\nSample:\n{dataset[0]['text'][:500]}...")

### 5a. (Optional) Augment with Real GRID Data

If you have a GRID database connection, this cell pulls real signals, anomalies, and filings to augment the synthetic training set. Skip if running on Colab without DB access.

In [ ]:
# Augment with real data from GRID DB (skip if no DB access)
import os

AUGMENT_FROM_DB = os.environ.get("DATABASE_URL") or os.path.exists("/content/grid/.env")

if AUGMENT_FROM_DB:
    try:
        from dotenv import load_dotenv
        load_dotenv("/content/grid/.env")
        
        from sqlalchemy import create_engine, text
        db_url = os.environ.get(
            "DATABASE_URL",
            f"postgresql://{os.environ.get('DB_USER', 'grid')}:{os.environ.get('DB_PASSWORD', '')}@"
            f"{os.environ.get('DB_HOST', 'localhost')}:{os.environ.get('DB_PORT', '5432')}/"
            f"{os.environ.get('DB_NAME', 'griddb')}"
        )
        engine = create_engine(db_url)
        
        if TASK == "signal_classifier":
            with engine.connect() as conn:
                rows = conn.execute(text(
                    "SELECT name, family, description FROM feature_registry "
                    "WHERE description IS NOT NULL AND LENGTH(description) > 20 LIMIT 50"
                )).fetchall()
            
            from gemma.training.config import TASK_SYSTEM_PROMPTS, TaskType
            sys_prompt = TASK_SYSTEM_PROMPTS[TaskType.SIGNAL_CLASSIFIER]
            augmented = 0
            for name, family, desc in rows:
                new_text = tokenizer.apply_chat_template([
                    {"role": "system", "content": sys_prompt},
                    {"role": "user", "content": f"Signal: {name} — {desc}"},
                    {"role": "assistant", "content": f"CATEGORY: {family or 'macro'}\nURGENCY: medium\nREASON: {desc[:80]}"},
                ], tokenize=False, add_generation_prompt=False).removeprefix("<bos>")
                dataset = dataset.add_item({"text": new_text, "conversations": []})
                augmented += 1
            print(f"Augmented with {augmented} real signals from feature_registry")
        
        print(f"Final dataset size: {len(dataset)} examples")
        engine.dispose()
    except Exception as e:
        print(f"DB augmentation skipped: {e}")
else:
    print("No DB access — using synthetic data only")

## 6. Train

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    eval_dataset=None,
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=WARMUP_STEPS,
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        logging_steps=1,
        optim="adamw_8bit",
        weight_decay=0.001,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir=f"outputs/{TASK}",
        report_to="none",
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
    ),
)

# Only train on model responses, mask system + user tokens
trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

print("Starting training...")
stats = trainer.train()
print(f"\nDone! Loss: {stats.training_loss:.4f}, Runtime: {stats.metrics.get('train_runtime', 0):.1f}s")

## 7. Test Inference

In [ ]:
from transformers import TextStreamer

# Test prompts per task
test_prompts = {
    "signal_classifier": (
        "Breaking: Federal Reserve announced emergency 50bp rate cut. "
        "Treasury yields dropping sharply. Equity futures surging."
    ),
    "anomaly_narrator": (
        "Feature: SPX_DAILY_RETURN\nValue: -6.8%\nExpected: -0.1%\n"
        "Z-score: -5.2\nPeriod: 2026-04-05\nContext: Largest single-day "
        "drop since March 2020. VIX above 40."
    ),
    "edgar_extractor": (
        "Extract these fields: company_name, filing_type, total_revenue, "
        "net_income, filing_date\n\nFiling text:\n"
        "AMAZON.COM INC\nFORM 10-Q\nQuarter ended March 31, 2026\n"
        "Net revenue: $155.7 billion\nNet income: $12.3 billion\n"
        "Filed: April 30, 2026"
    ),
    "knowledge_mapper": (
        "BlackRock increased its Bitcoin ETF holdings to $45B while simultaneously "
        "lobbying the SEC for spot Ethereum ETF approval. Larry Fink reversed his "
        "2017 anti-crypto stance. BlackRock also manages $2.3T in retirement assets "
        "for state pension funds."
    ),
}

system_prompt = TASK_SYSTEM_PROMPTS[task]
test_prompt = test_prompts[TASK]

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(
    messages, add_generation_prompt=True,
    tokenize=True, return_tensors="pt", return_dict=True,
).to("cuda")

print(f"=== {TASK} inference test ===")
print(f"Input: {test_prompt[:100]}...\n")
print("Response:")
_ = model.generate(
    **inputs, max_new_tokens=512,
    temperature=0.1, top_p=0.95,
    streamer=TextStreamer(tokenizer, skip_prompt=True),
)

## 8. Save Model

Save the LoRA adapter, then optionally export to GGUF for llama.cpp deployment.

In [ ]:
# Save LoRA adapter
lora_dir = f"outputs/{TASK}/lora"
model.save_pretrained(lora_dir)
tokenizer.save_pretrained(lora_dir)
print(f"LoRA adapter saved to {lora_dir}")

# Merge into full model (16-bit)
merged_dir = f"outputs/{TASK}/merged"
model.save_pretrained_merged(merged_dir, tokenizer)
print(f"Merged model saved to {merged_dir}")

In [ ]:
# Export GGUF for llama.cpp / Ollama deployment
if EXPORT_GGUF:
    gguf_dir = f"outputs/{TASK}/gguf"
    print(f"Exporting GGUF ({GGUF_QUANT})...")
    model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method=GGUF_QUANT)
    print(f"GGUF saved to {gguf_dir}")
    
    # Show the output file
    import glob
    gguf_files = glob.glob(f"{gguf_dir}/*.gguf")
    for f in gguf_files:
        size_mb = os.path.getsize(f) / (1024 * 1024)
        print(f"  {f} ({size_mb:.0f} MB)")

## 9. (Optional) Push to HuggingFace Hub

In [ ]:
# Push to HuggingFace Hub (uses HF_TOKEN from cell 4)
HF_USERNAME = "stepdadfinance"

if HF_TOKEN and EXPORT_GGUF:
    repo_name = f"{HF_USERNAME}/grid-gemma4-{TASK}"
    print(f"Pushing to {repo_name}...")
    model.push_to_hub_gguf(
        repo_name,
        tokenizer,
        quantization_method=GGUF_QUANT,
        token=HF_TOKEN,
    )
    print(f"Pushed to https://huggingface.co/{repo_name}")
else:
    if not HF_TOKEN:
        print("HF_TOKEN not set — skipping push. Set HF_API_KEY in .env or HF_TOKEN env var.")
    if not EXPORT_GGUF:
        print("GGUF export disabled — nothing to push.")

## 10. Download GGUF for Deployment

After training, download the GGUF file and deploy with llama.cpp:

```bash
# On your GRID server:
llama-server -m gemma-4-e4b-signal-classifier.gguf --port 8082 --threads 4
llama-server -m gemma-4-e4b-anomaly-narrator.gguf --port 8083 --threads 4
llama-server -m gemma-4-e4b-edgar-extractor.gguf --port 8084 --threads 4
llama-server -m gemma-4-e4b-knowledge-mapper.gguf --port 8085 --threads 4
```

The GRID `gemma/micro.py` pool will auto-connect to these endpoints.

The **Knowledge Mapper** model generates wiki-style entries with `[[backlinks]]` — pipe its output into a graph DB or markdown wiki to build a living knowledge graph of GRID's intelligence network.

In [ ]:
# Download from Colab
if EXPORT_GGUF:
    try:
        from google.colab import files
        for f in gguf_files:
            print(f"Downloading {f}...")
            files.download(f)
    except ImportError:
        print("Not running in Colab. Copy GGUF files manually from:")
        for f in gguf_files:
            print(f"  {f}")